# Quarterwatt – Colab runner

Electricity market data lake for 34 European bidding zones and Türkiye (2015 onwards). Data are stored in your own Google Drive under `MyDrive/Quarterwatt/`. See `docs/HANDBOOK.md` in the repository.

**Every session:** 1 → 2 → 3.

**Monthly update:** 1 → 2 → 3 → 4 (`qw.run()`) → 5 (`qw.finalize()`) → 9 (sync).

## 0 · Credentials (once)

The code contains no passwords or keys. Store them as **Colab Secrets**: click the key symbol in the left Colab bar → *Add new secret* → enter the name and value → switch on *Notebook access*. Names:

| Name | Used for |
|---|---|
| `ENTSOE_API_KEY` | ENTSO-E Transparency Platform (required) |
| `NTP_CLIENT_ID`, `NTP_CLIENT_SECRET` | netztransparenz.de (Germany: reBAP, ID-AEP, activated aFRR/mFRR) |
| `EPIAS_USERNAME`, `EPIAS_PASSWORD` | EPİAŞ Şeffaflık Platformu (Türkiye) |

Secrets belong to your Google account and work in every notebook once *Notebook access* is on. Cell 3 shows which ones are found.

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Get the code

Downloads the current `quarterwatt.py` from GitHub into `MyDrive/Quarterwatt/code/`. To pin a release, replace `main` by a tag such as `v1.0`.

In [ ]:
import os, requests
REPO_RAW = "https://raw.githubusercontent.com/serhanbasturk/quarterwatt/main/quarterwatt.py"
headers = {}
try:   # only for private copies: a GitHub token stored as Colab Secret GITHUB_TOKEN
    from google.colab import userdata
    if userdata.get("GITHUB_TOKEN"):
        headers["Authorization"] = "token " + userdata.get("GITHUB_TOKEN")
except Exception:
    pass
r = requests.get(REPO_RAW, headers=headers, timeout=60)
r.raise_for_status()
os.makedirs("/content/drive/MyDrive/Quarterwatt/code", exist_ok=True)
with open("/content/drive/MyDrive/Quarterwatt/code/quarterwatt.py", "w") as f:
    f.write(r.text)
print(r.text.splitlines()[1].strip())

## 3 · Load the code, check credentials and status

Downloads nothing.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content/drive/MyDrive/Quarterwatt/code')
import quarterwatt as qw
importlib.reload(qw)
qw.credentials_check()
qw.status()

### 3b · Diagnosis (only when needed)

`api_check()`: tests the ENTSO-E key with one request (use it if downloads fail).

In [ ]:
qw.api_check()

## 4 · Download / update

Fetches only what is missing plus the last 60 days. **If Colab disconnects:** run cells 1 → 2 → 3 → 4 again. Zones finished in the last 20 h are skipped and the interrupted zone continues where it stopped; recent months already refreshed today are not downloaded again. `qw.run(force=True)` processes every zone again.

You can also run in batches, e.g. `qw.run(qw.CORE)` first.

Months the source refuses with HTTP 400 are remembered (`.rejected`) and not asked again; `qw.RETRY_REJECTED = True` before `qw.run()` asks once more.

In [ ]:
# Quick test first (about 10 min): only the last ~13 months of one zone
# qw.START_DATE = "2025-10-01"; qw.run(["BE"])

qw.run()                  # all 35 zones
# qw.run(qw.CORE)          # only the 8 core countries
# qw.run(['DE', 'BE'])

## 5 · Check and combine into one panel

Loads exchange rates if needed, checks all zones and, if there is no FAIL, refreshes `Quarterwatt/data/panel/` (incl. `VERSION.json` and the licence column of `data_dictionary.csv`), `Quarterwatt/audit/` and the README. After a change of cleaning or currency rules: `qw.finalize(['PL', 'DE'])` first rebuilds these zones from the raw files (no downloads).

In [ ]:
result = qw.finalize()

## 6 · Reference data (once; again only with `force=True`)

BDEW standard load profiles (15 min, 2015–2027), load profiles of 50 German SME plants (Braeuer et al. 2020, Zenodo, CC BY 4.0) and German levies 2015–2026 → `Quarterwatt/data/reference/`.

In [ ]:
qw.update_reference()            
qw.load_reference('levies').tail()

## 7 · Use in analyses

In [ ]:
df  = qw.load_panel(['P_DA', 'Load_Actual'], zones=['DE', 'BE'])   # all zones, core columns
be  = qw.load_zone('BE')                                            # one zone, all columns
slp = qw.load_reference('slp')                                      # BDEW profiles (kW per MWh/a)
df.head()

## 8 · Maintenance: storage clean-up

Lists everything in `Quarterwatt/` that the code no longer uses (old cache versions, leftover backups, folders outside the layout). Data in use, outputs, audit, panel, reference data and exchange rates are never touched. Check the list, then run the second line.

In [ ]:
plan = qw.clean_storage()                 # only lists, changes nothing
# qw.clean_storage(delete=True)            # removes the listed items

## 9 · Sync to Drive (end of session)

Writes all pending changes to Google Drive and disconnects it. To continue afterwards, run cell 1 again.

In [ ]:
from google.colab import drive; drive.flush_and_unmount(); print("sync done")